Import modules

In [1]:
!pip install kagglehub
!pip install tensorflow
import kagglehub
import os
import tensorflow as tf


Download the dataset

In [ ]:

# Set download path
os.environ["KAGGLEHUB_CACHE"] = "dataset"

# Download from kaggle 
path = kagglehub.dataset_download("emmarex/plantdisease")    #Commented to prevent re-downloading 

# Followed my manually de-nesting the dataset 

 69% 453M/658M [01:43<00:41, 5.14MB/s] 

# Preprocess images by resizing, normalising, and applying suitable augmentation

In [ ]:
import tensorflow as tf

# 1. Load dataset directly from folder structure
dataset_path = "path/to/dataset"  # Directory with potato/, tomato/, etc.
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    directory=dataset_path,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=123
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    directory=dataset_path,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=123
)

# 2. Define Preprocessing & Augmentation Pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.1),            # Rotate +/- 10%
    tf.keras.layers.RandomBrightness(0.1),          # Adjust brightness
    tf.keras.layers.Rescaling(1./255),              # Scale pixel values [0, 255] -> [0, 1]
    tf.keras.layers.Normalization(                  # ImageNet mean & std normalization
        mean=[0.485, 0.456, 0.406],
        variance=[0.229**2, 0.224**2, 0.225**2]
    )
])

# 3. Apply transformations efficiently
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(buffer_size=tf.data.AUTOTUNE)

# Inspect classes
class_names = train_ds.class_names
print("Discovered classes:", class_names)